[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/uao/ia-geosciences-1/notebooks/M1_TD_TP_enonces.ipynb)

# Intelligence Artificielle & Géosciences — Master 1
## TD/TP : énoncés et corrections (5 séances × 2 h + exercices de cours + bonus)

| Séance | Thème | Durée |
|---|---|---|
| TD de cours | Type de problème, métriques, quiz | intégré au CM |
| S1 | Prise en main de Python et des données géoscientifiques | 2 h |
| S2 | Nettoyage (QA/QC) et visualisation exploratoire | 2 h |
| S3 | Régression supervisée : estimer le débit d’un forage | 2 h |
| S4 | Classification : forage productif vs improductif | 2 h |
| S5 | Clustering et usage raisonné de l’IA générative | 2 h |
| Bonus | Validation spatiale (au-delà des 20 h) | 1–2 h |

**Données** : `forages_socle_synthetique.csv` (dossier `M1_TP_donnees`) — 320 forages synthétiques réalistes de socle cristallin, contenant **volontairement** des valeurs manquantes et des codes erronés (−999).

| Colonne | Signification | Unité |
|---|---|---|
| id, x_km, y_km, zone | identifiant, position, zone | —, km |
| lithologie | Granite, Schiste, Migmatite, Volcano-sed | — |
| profondeur_m | profondeur du forage | m |
| epaisseur_alteration_m | épaisseur des altérites | m |
| resistivite_ohm_m | résistivité apparente (ERP) | Ω·m |
| sfi, magnitude, power | attributs de l’anomalie ERP | — |
| ohmic_area | aire ohmique (VES) | Ω·m² |
| debit_m3h | débit mesuré (**cible**) | m³/h |

> **Google Colab** : si le dossier `IA_Geosciences_M1` est dans votre Google Drive, mettez `UTILISER_DRIVE = True` dans la cellule ci-dessous ; sinon laissez `False` et choisissez le fichier CSV quand la fenêtre s’ouvre.
> **Sans Internet (Jupyter local)** : rien à modifier.

In [ ]:
# Configuration commune à toutes les séances
# ---------------- RÉGLAGES : seules lignes à modifier ----------------
UTILISER_DRIVE = False   # Colab : mettre True si le dossier est dans votre Google Drive
DOSSIER_DRIVE = '/content/drive/MyDrive/IA_Geosciences_M1'   # chemin du dossier dans Drive
# ---------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import os
NOM = 'forages_socle_synthetique.csv'
try:                                   # Sommes-nous sur Google Colab ?
    import google.colab
    SUR_COLAB = True
except ImportError:
    SUR_COLAB = False

if SUR_COLAB and UTILISER_DRIVE:       # Colab + Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    DOSSIER = DOSSIER_DRIVE
elif SUR_COLAB:                        # Colab sans Drive : téléchargement depuis le dépôt du cours
    DOSSIER = '/content'
    if not os.path.exists(f'{DOSSIER}/{NOM}'):
        try:
            import urllib.request
            urllib.request.urlretrieve(f'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/uao/ia-geosciences-1/data/{NOM}', f'{DOSSIER}/{NOM}')
        except Exception:                # pas d'accès au dépôt : téléversement direct
            from google.colab import files
            print('Choisissez le fichier', NOM, 'sur votre ordinateur...')
            files.upload()
else:                                  # Jupyter local, sans Internet
    DOSSIER = next((d for d in ['.', 'donnees', '../data', '../M1_TP_donnees'] if os.path.exists(f'{d}/{NOM}')), '.')

FICHIER = f'{DOSSIER}/{NOM}'
assert os.path.exists(FICHIER), f'Fichier introuvable : {FICHIER} (vérifier UTILISER_DRIVE et DOSSIER_DRIVE)'
print('Données :', FICHIER)
pd.set_option('display.width', 120)

---
# TD de cours (exercices intégrés au cours magistral)

## Exercice 1 (Module 1) — Tâche, Performance, Expérience
Selon la définition de T. Mitchell, une machine apprend si sa **performance P** sur une **tâche T** s’améliore avec l’**expérience E**.

Donner T, P et E pour :
1. la cartographie automatique des faciès à partir de diagraphies ;
2. la prévision du niveau de nappe le mois prochain.

## Exercice 2 (Module 2) — Quel type de problème ?
Pour chaque question, indiquer si l’apprentissage est **supervisé** ou non, et le **type de problème** (classification binaire / multi-classes, régression, clustering, détection d’anomalies). Justifier par la nature de y.

| # | Question géoscientifique |
|---|---|
| 1 | Ce futur forage sera-t-il productif (> 1 m³/h) ? |
| 2 | Quelle sera la perméabilité de ce niveau réservoir ? |
| 3 | Existe-t-il des familles géochimiques dans 500 analyses XRF ? |
| 4 | Quel faciès (grès, argile, calcaire) à chaque mètre du puits ? |
| 5 | Quelles mesures de résistivité sont suspectes sur ce profil ? |
| 6 | Quel niveau de nappe le mois prochain ? |

## Exercice 3 (Module 4) — Calculer les métriques de classification
Un modèle de prédiction de forages productifs donne sur le jeu de test : **VP = 40, FP = 10, FN = 20, VN = 30**.

1. Calculer l’accuracy, la précision, le rappel et le F1-score.
2. Interpréter FP et FN en termes de conséquences sur le terrain.
3. Si 95 % des forages d’une région étaient secs, quelle serait l’accuracy du modèle « toujours sec » ? Quel est son rappel ?

In [ ]:
VP, FP, FN, VN = 40, 10, 20, 30
# À COMPLÉTER : accuracy, precision, rappel, f1


## Exercice 4 — Quiz de cours (réponses courtes)
1. Un LLM est-il du Deep Learning ?
2. Pourquoi un arbre de décision très profond surapprend-il ?
3. Que « vote » une forêt aléatoire ?
4. Quel est le rôle de la fonction d’activation dans un réseau de neurones ?
5. Vous disposez de 200 forages répartis dans 5 villages. Comment construisez-vous votre jeu de test ?

---
# Séance 1 — Prise en main de Python et des données géoscientifiques (2 h)
**Objectifs** : découvrir Jupyter/Colab ; charger, inspecter et manipuler un tableau de forages avec pandas.

## Étapes guidées
1. Importer les bibliothèques et charger le fichier.
2. Premières statistiques descriptives.
3. Filtrer les forages (lithologie, seuil de débit).
4. Agréger par lithologie (`groupby`).

In [ ]:
df = pd.read_csv(FICHIER)
print(df.shape)          # (lignes, colonnes)
df.head()

In [ ]:
df.info()

In [ ]:
df[['epaisseur_alteration_m', 'resistivite_ohm_m', 'sfi', 'debit_m3h']].describe().round(2)

**Questions d’observation**
- Pourquoi `count` diffère-t-il selon les colonnes ?
- Que signifie un minimum de −999 Ω·m ?
- Le débit médian (50 %) est-il proche de la moyenne ? Qu’en déduire ?

In [ ]:
# Filtrage et agrégation (étapes 3–4)
granites = df[df['lithologie'] == 'Granite']
exploitables = df[df['debit_m3h'] > 1]
print('part de forages exploitables :', round(len(exploitables) / len(df), 3))
df.groupby('lithologie')['debit_m3h'].mean().round(2).sort_values()

## Exercices de la séance 1
1. Combien de forages compte chaque zone ? Laquelle a le plus fort débit moyen ?
2. Quelle proportion de forages en Schiste dépasse 1 m³/h ? Comparer au Granite.
3. Afficher les 10 forages les plus productifs.
4. Calculer la médiane de l’épaisseur d’altération par lithologie.
5. *(Bonus)* Tracer l’histogramme des débits.

In [ ]:
# Exercice 1
# Exercice 2
# Exercice 3
# Exercice 4
# Exercice 5 (bonus)


---
# Séance 2 — Nettoyage (QA/QC) et visualisation exploratoire (2 h)
**Objectifs** : repérer et traiter erreurs, valeurs manquantes et valeurs aberrantes ; explorer les relations entre variables géophysiques et débit.

## Exercice 2.1 — Diagnostiquer
1. Compter les valeurs manquantes par colonne.
2. Compter les résistivités physiquement impossibles (≤ 0).
3. Tracer le boxplot de la résistivité brute : que remarquez-vous ?

In [ ]:
# À COMPLÉTER


## Exercice 2.2 — Nettoyer
1. Créer une copie `df_clean` ; remplacer les résistivités ≤ 0 par NaN.
2. Imputer les valeurs manquantes par la **médiane** (résistivité, épaisseur d’altération).
3. Vérifier qu’il ne reste aucune valeur manquante.
4. *(Discussion)* La médiane globale est-elle adaptée ? Proposer une imputation **par lithologie**.

In [ ]:
# À COMPLÉTER


## Exercice 2.3 — Corrélations
Calculer et afficher la matrice de corrélation (Pearson) entre profondeur, épaisseur d’altération, résistivité, sfi, aire ohmique et débit. Quelle variable est la plus corrélée au débit ?

In [ ]:
# À COMPLÉTER


## Exercice 2.4 — Nuage de points et transformation logarithmique
1. Tracer le débit en fonction de la résistivité (échelle log en x), coloré par le sfi.
2. Créer `log_rho = log10(résistivité)` et comparer les histogrammes brut / log.
3. Les forages les plus productifs ont-ils une résistivité faible, moyenne ou forte ? Pourquoi, géologiquement ?

In [ ]:
# À COMPLÉTER


---
# Séance 3 — Régression supervisée : estimer le débit d’un forage (2 h)
**Objectifs** : formuler X → y, entraîner un modèle linéaire et un modèle à base d’arbres, évaluer avec MAE, RMSE, R².

## Exercice 3.1 — Définir X, y et partitionner
Utiliser les variables `epaisseur_alteration_m, log_rho, sfi, magnitude, power, ohmic_area` et la cible `debit_m3h`. Découper en 80 % / 20 % avec `random_state=0`.

*Question* : pourquoi ne pas utiliser `id`, `x_km`, `y_km` ou `profondeur_m` ?

In [ ]:
from sklearn.model_selection import train_test_split
# À COMPLÉTER


## Exercice 3.2 — Entraîner et évaluer
Entraîner une régression linéaire, un arbre de décision (`max_depth=4`) et une forêt aléatoire (300 arbres, `min_samples_leaf=3`). Calculer MAE, RMSE et R² sur le test et présenter un tableau comparatif.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# À COMPLÉTER


## Exercice 3.3 — Surapprentissage
Faire varier `max_depth` de l’arbre de 1 à 15 et tracer le R² d’entraînement et de test. À partir de quelle profondeur l’arbre surapprend-il ?

In [ ]:
# À COMPLÉTER


## Exercice 3.4 — Analyse critique « débit réel vs prédit »
Tracer le débit prédit en fonction du débit réel pour la régression linéaire et la forêt (avec la diagonale). Rédiger 10 lignes : quel modèle est le meilleur et selon quelle métrique ? où sont les plus grosses erreurs ? quelles données de terrain amélioreraient le modèle ?

In [ ]:
# À COMPLÉTER


---
# Séance 4 — Classification : forage productif vs improductif (2 h)
**Objectifs** : transformer un problème continu en classes décisionnelles, entraîner une forêt aléatoire, interpréter la matrice de confusion.

## Exercice 4.1 — Variable binaire et entraînement
1. Créer `productif = 1` si débit > 1 m³/h, 0 sinon ; afficher la proportion de chaque classe.
2. Découper (75/25, `stratify=y`, `random_state=1`).
3. Entraîner une régression logistique et une forêt aléatoire (300 arbres).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
# À COMPLÉTER


## Exercice 4.2 — Évaluer la classification
Pour les deux modèles : accuracy, précision, rappel, F1 (classe productif) et matrice de confusion de la forêt. Compléter le tableau comparatif.

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score
# À COMPLÉTER


## Exercice 4.3 — Le coût d’un faux positif (TD de réflexion)
1. Que coûte un forage sec à une commune ou à une ONG ? Que coûte un village sans point d’eau ?
2. Utiliser un seuil de décision de **0,7** au lieu de 0,5 (`predict_proba`). Comment évoluent précision et rappel ?
3. Tracer précision et rappel en fonction du seuil. Quel seuil recommanderiez-vous ? Qui doit prendre la décision finale ?

In [ ]:
# À COMPLÉTER


## Exercice 4.4 — Importance des variables
Afficher l’importance des variables de la forêt. Les variables dominantes ont-elles un sens hydrogéologique ? Pourquoi la résistivité seule pèse-t-elle moins ?

In [ ]:
# À COMPLÉTER


---
# Séance 5 — Clustering et usage raisonné de l’IA générative (2 h)
## Partie A — Regrouper des signatures géoélectriques (K-Means)
### Exercice 5.1
1. Standardiser `log_rho`, `epaisseur_alteration_m` et `sfi`.
2. Tracer la méthode du coude (K = 1 à 9).
3. Appliquer K-Means avec K = 4 et ajouter la colonne `cluster`.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
# À COMPLÉTER


### Exercice 5.2 — Les clusters correspondent-ils à la géologie ?
1. Croiser `cluster` et `lithologie` (`pd.crosstab`).
2. Comparer le débit moyen par cluster.
3. Les clusters ont-ils une cohérence spatiale ? Que changerait K = 3 ?

In [ ]:
# À COMPLÉTER


## Partie B — Usage raisonné d’un assistant IA (LLM)
### Exercice 5.3 — Un script contenant des erreurs
Le script suivant doit prédire le débit à partir du sfi et de l’épaisseur d’altération, mais il ne fonctionne pas.

```python
import pandas
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv('forages_socle_synthetique.csv')
X = df[['sfi', 'epaisseur_alteration_m']]
y = df['debit']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=80)
modele = RandomForestRegressor().fit(X_train, y_train)
y_pred = modele.predict(X_train)
print(r2_score(y_test, y_pred))
```
1. Repérer **vous-même** au moins deux erreurs (exécuter, lire les messages).
2. Trouver l’erreur **logique** qui ne produit aucun message.
3. Seulement ensuite, demander l’aide d’un LLM avec un prompt structuré (Rôle, Contexte, Tâche, Contraintes, Format).

### Exercice 5.4 — Rédiger un prompt structuré, puis évaluer la réponse
Rédiger le prompt, puis évaluer la réponse du LLM avec la grille : exécution, exhaustivité (dont l’erreur logique), compréhension, fidélité à la méthode, cohérence du R² avec la séance 3, traçabilité.

---
# Bonus (au-delà des 20 h) — La validation spatiale en pratique
### Exercice B.1
Avec une forêt aléatoire utilisant les variables de la séance 3 **plus** `x_km, y_km` :
1. Calculer le R² moyen en validation croisée **aléatoire** (`KFold(5, shuffle=True)`).
2. Calculer le R² moyen en validation **par zone** (`GroupKFold(4)`, groupes = `zone`).
3. Expliquer l’écart.

In [ ]:
from sklearn.model_selection import KFold, GroupKFold, cross_val_score
# À COMPLÉTER


---
## Évaluation proposée
- **QCM (30 %)** : vocabulaire, types de problèmes, métriques, surapprentissage, validation spatiale, LLM.
- **Mini-projet (50 %)** : notebook commenté QA/QC → modèle → métriques ; baseline vs forêt ; validation aléatoire et par zone.
- **Fiche d’interprétation (20 %)** : 1 page, métrique + protocole + réalité géologique, limites, déclaration d’usage d’IA générative.